# EH2025 – Caso 3: Discriminación y Seguridad Ciudadana

**Pregunta:** ¿Qué factores sociodemográficos y geográficos están asociados con una mayor percepción de discriminación y victimización en Bolivia?

**Fuente:** INE Bolivia, Encuesta de Hogares 2025 (EH2025), ANDA: http://anda.ine.gob.bo/index.php/catalog/256.

**Datos requeridos:** `data/raw/EH2025_Discriminacion.sav` y `data/raw/EH2025_Persona.sav`. Los microdatos no deben publicarse en GitHub.

Ejecutar las celdas en orden. Los modelos son exploratorios y no sustituyen estimaciones que incorporen el diseño muestral completo.


## 0. Instalación e importaciones

In [ ]:
# En Colab, si hace falta:
# !pip -q install duckdb pyreadstat ydata-profiling plotly folium scikit-learn shap xgboost statsmodels

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import duckdb
import pyreadstat
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             f1_score, precision_score, recall_score, balanced_accuracy_score)
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

ROOT = Path(".") if Path("data/raw").exists() else (Path("..") if Path("../data/raw").exists() else Path("."))
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
for sub in ["reports","figures","dashboard","models"]:
    (OUT/sub).mkdir(parents=True, exist_ok=True)

disc_path = RAW / "EH2025_Discriminacion.sav"
person_path = RAW / "EH2025_Persona.sav"
assert disc_path.exists() and person_path.exists(), (
    "Coloca EH2025_Discriminacion.sav y EH2025_Persona.sav en data/raw/"
)


## 1. Ingesta SAV, DuckDB y control de calidad

In [ ]:
disc, meta_disc = pyreadstat.read_sav(disc_path, apply_value_formats=False)
person, meta_person = pyreadstat.read_sav(person_path, apply_value_formats=False)
print("Módulo discriminación:", disc.shape, "| módulo persona:", person.shape)
print("Duplicados clave en discriminación:", disc.duplicated(["folio","nro"]).sum())

con = duckdb.connect()
con.register("disc", disc)
con.register("persona", person)
sql = '''
SELECT d.*, p.s01a_02, p.s01a_03, p.s01a_09, p.s01a_08, p.s01a_10
FROM disc d LEFT JOIN persona p
 ON CAST(d.folio AS VARCHAR)=CAST(p.folio AS VARCHAR)
AND CAST(d.nro AS INTEGER)=CAST(p.nro AS INTEGER)
'''
df = con.execute(sql).df()
assert len(df) == len(disc), "El JOIN alteró el número de registros; revisar claves."
print("Filas tras JOIN:", len(df))


## 2. EDA automatizado HTML

In [ ]:
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(disc, title="EH2025 – Perfil del módulo de discriminación",
                            explorative=True, minimal=True)
    profile.to_file(OUT/"reports"/"eda_ydata_profiling.html")
    print("Reporte guardado:", OUT/"reports"/"eda_ydata_profiling.html")
except Exception as e:
    print("YData-Profiling no pudo ejecutarse. Alternativa Sweetviz:", repr(e))
    try:
        import sweetviz as sv
        report = sv.analyze(disc)
        report.show_html(str(OUT/"reports"/"eda_sweetviz.html"), open_browser=False)
    except Exception as e2:
        print("Sweetviz tampoco pudo ejecutarse:", repr(e2))


## 3. Ingeniería de variables objetivo

**Discriminación:** positiva si al menos un ítem `s09a_01a`–`s09a_01l` es 1; negativa si todos los ítems están clasificados como 1 o 2 y no hay positivos; el resto se deja sin clasificar.

**Victimización:** se usan provisionalmente las etiquetas leídas del SAV: códigos 1–7 = hechos delictivos; código 8 = “Ninguno”. Esta codificación difiere del catálogo PDF consultado; confirmar con INE.

In [ ]:
disc_vars = [f"s09a_01{x}" for x in "abcdefghijkl"]
assert all(c in df.columns for c in disc_vars + ["s09b_02a","s09b_02b","s09b_01"])
has_yes = df[disc_vars].eq(1).any(axis=1)
all_classified = df[disc_vars].isin([1,2]).all(axis=1)
df["discriminacion_bin"] = np.where(has_yes, 1,
    np.where(all_classified & ~has_yes, 0, np.nan))

crime = df[["s09b_02a","s09b_02b"]].isin([1,2,3,4,5,6,7]).any(axis=1)
no_crime = df["s09b_02a"].eq(8) & (df["s09b_02b"].eq(8) | df["s09b_02b"].isna())
df["victimizacion_bin"] = np.select([crime,no_crime],[1,0],default=np.nan)

print("Discriminación:")
display(df["discriminacion_bin"].value_counts(dropna=False))
print("Victimización:")
display(df["victimizacion_bin"].value_counts(dropna=False))


## 4. Indicadores departamentales ponderados

In [ ]:
deptos = {1:"Chuquisaca",2:"La Paz",3:"Cochabamba",4:"Oruro",5:"Potosí",
          6:"Tarija",7:"Santa Cruz",8:"Beni",9:"Pando"}
df["departamento"] = pd.to_numeric(df["depto"],errors="coerce").map(deptos)
df["ponderador"] = pd.to_numeric(df["ponderador"],errors="coerce")
def wrate(g, col):
    x, w = g[col], g["ponderador"]
    ok = x.notna() & w.notna() & (w>0)
    return np.average(x[ok].astype(float), weights=w[ok]) if ok.any() else np.nan

rows=[]
for name,g in df.groupby("departamento"):
    row={"departamento":name,"Muestra":len(g)}
    for label,codev in [("Muy_inseguro",1),("Inseguro",2),("Seguro",3),("Muy_seguro",4)]:
        ok = g["ponderador"].notna() & (g["ponderador"] > 0) & g["s09b_01"].notna()
        row[label] = np.average(
            (g.loc[ok, "s09b_01"] == codev).astype(float),
            weights=g.loc[ok, "ponderador"]
        ) * 100
    row["Victimizacion"]=wrate(g,"victimizacion_bin")*100
    row["Discriminacion"]=wrate(g,"discriminacion_bin")*100
    rows.append(row)
tabla_depto=pd.DataFrame(rows)
tabla_depto["Inseguridad_percibida"]=tabla_depto["Muy_inseguro"]+tabla_depto["Inseguro"]
display(tabla_depto.round(2))
tabla_depto.to_csv(OUT/"departamento_indicadores_ponderados.csv",index=False)


## 5. Cuatro visualizaciones interactivas Plotly

In [ ]:
fig1=px.bar(tabla_depto,x="departamento",y="Discriminacion",
 title="Discriminación percibida por departamento",
 labels={"departamento":"Departamento","Discriminacion":"Porcentaje ponderado"})
fig1.write_html(OUT/"figures"/"01_discriminacion_departamento.html"); fig1.show()

fig2=px.bar(tabla_depto,x="departamento",y=["Muy_inseguro","Inseguro","Seguro","Muy_seguro"],
 barmode="group",title="Percepción de seguridad nocturna por departamento",
 labels={"value":"Porcentaje ponderado","variable":"Percepción"})
fig2.write_html(OUT/"figures"/"02_seguridad_departamento.html"); fig2.show()

fig3=px.scatter(tabla_depto,x="Inseguridad_percibida",y="Victimizacion",
 text="departamento",size="Muestra",title="Perfiles departamentales de seguridad",
 labels={"Inseguridad_percibida":"Inseguridad percibida (%)","Victimizacion":"Victimización (%)"})
fig3.write_html(OUT/"figures"/"03_perfiles_departamentales.html"); fig3.show()

fig4=px.scatter(df.dropna(subset=["s01a_03","s01a_02","discriminacion_bin"]),
 x="s01a_03",y="s01a_02",color="discriminacion_bin",
 hover_data=["departamento","area"],opacity=.45,
 title="Edad, sexo y clasificación de discriminación",
 labels={"s01a_03":"Edad","s01a_02":"Código de sexo","discriminacion_bin":"Discriminación"})
fig4.write_html(OUT/"figures"/"04_edad_sexo_discriminacion.html"); fig4.show()


## 6. Machine Learning: validación cruzada estratificada

In [ ]:
features=["s01a_03","s01a_02","depto","area"]
cat=["s01a_02","depto","area"]; num=["s01a_03"]
prep=ColumnTransformer([
 ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                  ("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat),
 ("num",SimpleImputer(strategy="median"),num)
])
models={
 "RandomForest":RandomForestClassifier(n_estimators=300,min_samples_leaf=4,
   class_weight="balanced",random_state=42,n_jobs=-1),
 "ExtraTrees":ExtraTreesClassifier(n_estimators=300,min_samples_leaf=4,
   class_weight="balanced",random_state=42,n_jobs=-1)
}
try:
 from xgboost import XGBClassifier
 models["XGBoost"]=XGBClassifier(n_estimators=250,max_depth=3,learning_rate=.04,
   subsample=.85,colsample_bytree=.85,eval_metric="logloss",random_state=42,n_jobs=-1)
except Exception as e:
 print("XGBoost opcional no disponible:",e)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring={"f1":"f1","precision":"precision","recall":"recall",
         "roc_auc":"roc_auc","balanced_accuracy":"balanced_accuracy"}
def cv_compare(target):
 d=df[features+[target]].dropna(subset=[target])
 X=d[features]; y=d[target].astype(int); rows=[]
 for name,est in models.items():
  pipe=Pipeline([("prep",prep),("model",est)])
  sc=cross_validate(pipe,X,y,cv=cv,scoring=scoring,n_jobs=-1)
  row={"Objetivo":target,"Modelo":name}
  for metric in scoring:
   row[metric+"_media"]=sc["test_"+metric].mean()
   row[metric+"_sd"]=sc["test_"+metric].std()
  rows.append(row)
 return pd.DataFrame(rows).sort_values("f1_media",ascending=False)
cv_disc=cv_compare("discriminacion_bin")
cv_vict=cv_compare("victimizacion_bin")
display(cv_disc); display(cv_vict)
pd.concat([cv_disc,cv_vict]).to_csv(OUT/"modelos_validacion_cruzada.csv",index=False)


## 7. Evaluación hold-out, matriz de confusión e importancia por permutación

In [ ]:
def holdout_and_importance(target):
 d=df[features+[target]].dropna(subset=[target]).copy()
 X,y=d[features],d[target].astype(int)
 Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
 pipe=Pipeline([("prep",prep),("model",ExtraTreesClassifier(
     n_estimators=300,min_samples_leaf=4,class_weight="balanced",
     random_state=42,n_jobs=-1))])
 pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); prob=pipe.predict_proba(Xte)[:,1]
 print(target, "AUC:",roc_auc_score(yte,prob),"F1:",f1_score(yte,pred))
 print(classification_report(yte,pred,zero_division=0))
 print("Matriz de confusión:\n",confusion_matrix(yte,pred))
 pi=permutation_importance(pipe,Xte,yte,n_repeats=10,random_state=42,scoring="f1")
 imp=pd.DataFrame({"variable":features,"importancia":pi.importances_mean,
                   "sd":pi.importances_std}).sort_values("importancia",ascending=False)
 display(imp)
 fig=px.bar(imp,x="importancia",y="variable",orientation="h",
            title=f"Importancia por permutación – {target}")
 fig.write_html(OUT/"figures"/f"05_importancia_{target}.html")
 return pipe,imp
model_disc,imp_disc=holdout_and_importance("discriminacion_bin")
model_vict,imp_vict=holdout_and_importance("victimizacion_bin")


## 8. Explainability SHAP (opcional)

In [ ]:
try:
 import shap
 # SHAP para árbol ExtraTrees. El muestreo se limita para controlar memoria.
 Xs=df[features].dropna().sample(min(300,df[features].dropna().shape[0]),random_state=42)
 Xt=model_disc.named_steps["prep"].transform(Xs)
 explainer=shap.TreeExplainer(model_disc.named_steps["model"])
 sv=explainer.shap_values(Xt)
 shap.summary_plot(sv[1] if isinstance(sv,list) else sv, Xt, show=False)
 plt.tight_layout()
 plt.savefig(OUT/"figures"/"06_shap_discriminacion.png",dpi=160,bbox_inches="tight")
 plt.show()
except Exception as e:
 print("SHAP no pudo generarse en este entorno:",repr(e),
       "La importancia por permutación queda como explicación complementaria.")


## 9. Clustering departamental ponderado

In [ ]:
Xc=tabla_depto[["Inseguridad_percibida","Victimizacion"]].copy()
Z=StandardScaler().fit_transform(Xc)
sil_rows=[]
for k in [2,3,4]:
 km0=KMeans(n_clusters=k,random_state=42,n_init=20)
 lab=km0.fit_predict(Z)
 sil_rows.append({"clusters":k,"silhouette":silhouette_score(Z,lab)})
sil=pd.DataFrame(sil_rows); display(sil)
best_k=int(sil.sort_values("silhouette",ascending=False).iloc[0]["clusters"])
km=KMeans(n_clusters=best_k,random_state=42,n_init=20)
tabla_depto["Cluster"]=km.fit_predict(Z)
display(tabla_depto[["departamento","Inseguridad_percibida","Victimizacion","Cluster"]])
figc=px.scatter(tabla_depto,x="Inseguridad_percibida",y="Victimizacion",
 text="departamento",color=tabla_depto["Cluster"].astype(str),
 title=f"Clustering departamental K-Means (k={best_k})")
figc.write_html(OUT/"figures"/"07_clustering_departamental.html"); figc.show()


## 10. Segmentación exploratoria de personas (K-Means)

La consigna solicita segmentación de individuos u hogares. Esta sección crea perfiles exploratorios de personas con edad, sexo, departamento y área. Se usa una muestra reproducible para limitar memoria. Los clusters no son categorías oficiales ni causales.


In [ ]:
from sklearn.decomposition import PCA
ind = df[["s01a_03", "s01a_02", "depto", "area"]].dropna().copy()
if len(ind) > 6000:
    ind = ind.sample(6000, random_state=42)
pre_ind = ColumnTransformer([
    ("num", StandardScaler(), ["s01a_03"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["s01a_02", "depto", "area"])
])
Zi = pre_ind.fit_transform(ind)
km_ind = KMeans(n_clusters=3, random_state=42, n_init=20)
ind["cluster_persona"] = km_ind.fit_predict(Zi).astype(str)
display(ind.groupby("cluster_persona").agg(
    personas=("s01a_03", "size"), edad_media=("s01a_03", "mean"),
    proporcion_mujeres=("s01a_02", lambda x: (x==2).mean()*100),
    proporcion_rural=("area", lambda x: (x==2).mean()*100)
).round(2))
Z_dense = Zi.toarray() if hasattr(Zi, "toarray") else Zi
coords = PCA(n_components=2, random_state=42).fit_transform(Z_dense)
plot_ind = ind.copy(); plot_ind["PCA1"] = coords[:,0]; plot_ind["PCA2"] = coords[:,1]
fig_ind = px.scatter(plot_ind, x="PCA1", y="PCA2", color="cluster_persona",
    hover_data=["s01a_03", "s01a_02", "depto", "area"],
    title="Perfiles exploratorios de personas: K-Means + PCA")
fig_ind.write_html(OUT/"figures"/"08_clustering_personas_pca.html")
fig_ind.show()


## 10. Mapa Folium

Para construir un mapa departamental interactivo se requiere un GeoJSON con límites de los nueve departamentos. El código siguiente usa una URL de GeoJSON pública; si el proveedor cambia la ruta o bloquea el acceso, se debe guardar el GeoJSON localmente y ajustar `geojson_path`. El mapa usa fondo neutro y no depende de teselas de OSM.

In [ ]:
import folium, requests, unicodedata, json
from pathlib import Path
geojson_path=ROOT/"data"/"raw"/"bolivia_departamentos.geojson"
geojson_url="https://raw.githubusercontent.com/YoViajo/geodatos/master/limites/bol_lim_dpto.json"
if geojson_path.exists():
 geo=json.loads(geojson_path.read_text(encoding="utf-8"))
else:
 resp=requests.get(geojson_url,timeout=60); resp.raise_for_status(); geo=resp.json()

def norm(s):
 s=str(s).strip().lower()
 return "".join(c for c in unicodedata.normalize("NFD",s)
                if unicodedata.category(c)!="Mn")
lookup={norm(r["departamento"]):r for _,r in tabla_depto.iterrows()}
for feat in geo["features"]:
 for val in feat["properties"].values():
  if isinstance(val,str) and norm(val) in lookup:
   r=lookup[norm(val)]
   feat["properties"].update({"departamento_std":r["departamento"],
      "Cluster":int(r["Cluster"]),
      "Inseguridad":round(float(r["Inseguridad_percibida"]),2),
      "Victimizacion":round(float(r["Victimizacion"]),2)})
   break
found={f["properties"]["departamento_std"] for f in geo["features"]
       if "departamento_std" in f["properties"]}
assert len(found)==9, f"GeoJSON unió {len(found)} departamentos: {found}"
m=folium.Map(location=[-17,-64.5],zoom_start=5,tiles=None)
colors={0:"#D95F02",1:"#1B9E77",2:"#7570B3",3:"#E7298A"}
folium.GeoJson(geo,name="Perfiles EH2025",
 style_function=lambda f:{"fillColor":colors.get(f["properties"]["Cluster"],"#999999"),
 "color":"#333","weight":1.4,"fillOpacity":.65},
 tooltip=folium.GeoJsonTooltip(fields=["departamento_std","Cluster","Inseguridad","Victimizacion"],
 aliases=["Departamento","Cluster","Inseguridad percibida (%)","Victimización (%)"])
).add_to(m)
m.save(str(OUT/"dashboard"/"mapa_seguridad_bolivia.html"))
display(m)


## 11. Exportación y notas finales

In [ ]:
tabla_depto.to_csv(OUT/"departamento_indicadores_ponderados.csv",index=False)
print("Salidas guardadas en:",OUT.resolve())
print("Antes de publicar: comprobar outputs, documentar fecha de descarga y no subir microdatos.")
